# Object-Oriented Market Making

Audience: Python learners preparing for quant interviews. Prerequisites: basic Python and pandas.

Goals: Quote and trade objects, state management, histories.

Outline: setup, short tasks, checks, pitfalls. Run cells from top to bottom. All market data here is synthetic.


In [1]:
from pathlib import Path
import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
from market_simulator import (MarketMaker, ClientModel, MarketEnvironment,
    QuotedTrade, CompletedTrade, generate_market_data, generate_rfqs, run_simulation, make_quote)
from market_simulator.risk import beta, historical_volatility

# Resolve synthetic data from the installed local package, independent of kernel CWD.
import market_simulator
DATA = Path(market_simulator.__file__).resolve().parents[1] / 'data'
prices = pd.read_csv(DATA / 'market_prices.csv', parse_dates=['timestamp'])
requests = pd.read_csv(DATA / 'client_rfqs.csv', parse_dates=['timestamp'])
print(prices.head(2).to_string(index=False))


                timestamp ticker   mid  market_bid  market_ask
2026-01-02 09:30:00+00:00   ALFA 100.0        99.9       100.1
2026-01-02 09:30:00+00:00   BETA 100.0        99.9       100.1


## Quote objects

Build a quote with `make_quote` for ticker ALFA, quantity 10, mid 100, timestamp 0 and full spread 0.02. Return its bid and offer as a tuple.


In [3]:
def object_quote() -> tuple[float, float]:
    # TODO
    quoted_trade = make_quote('ALFA', 10, 100, 0, 0.02)
    return (quoted_trade.bid_price, quoted_trade.offer_price)

object_quote()

(99.0, 101.0)

## Execute a maker buy

Create a MarketMaker with cash 1000 and limit 20. Add a 10-share quote at bid 99, ask 101, then execute a maker buy at the bid. Return (inventory, cash, history_length).


In [4]:
def buy_example() -> tuple[int, float, int]:
    # TODO
    market_maker = MarketMaker(1000, 20)
    quoted_trade = make_quote('ALFA', 10, 100, 0, 0.02)
    market_maker.add_quoted_trade(quoted_trade)
    market_maker.add_trade(CompletedTrade('ALFA', 10, quoted_trade.bid_price, 'buy', 100, quoted_trade.bid_price, quoted_trade.offer_price, 0))
    return (market_maker.position('ALFA'), market_maker.cash, len(market_maker.completed_trades))

buy_example()

(10, 10.0, 1)

## Mark inventory

Given a maker that already traded and a current mark dictionary, return its total P&L using mark_to_market.


In [7]:
def total_pnl(maker: MarketMaker, marks: dict[str, float]) -> float:
    # TODO
    return maker.mark_to_market(marks)['total_pnl']

m = MarketMaker(initial_cash=1000)
q = make_quote('A', 1, 100, 0)
m.add_trade(CompletedTrade('A', 1, 99, 'buy', 100, 99, 101, 0))
total_pnl(m, {'A':100})

1.0

## Extension
Try a different random seed or add a transaction fee. Explain how your answer changes.